# PRT-CP: Physics-Residual Transformer with Conformal Prediction

> **Paper**: *Trustworthy Virtual Sensing via Physics-Residual Transformers and Conformal Prediction*
> **Authors**: Bin Wang, Enrico Zio
> **Repository**: https://github.com/WHUTBIN/Papers/tree/main/PRT-CP

---

This notebook reproduces the LPG case study from the paper. It demonstrates the PRT-CP framework: a Transformer model with a physics-informed residual branch, calibrated with split conformal prediction, evaluated by the Composite Trustworthiness Index (CTI).

**Data**: The original LPG dataset is proprietary and cannot be released. Replace `DATA_PATH` below with your own file. The expected columns are:

| Column | Description |
|---|---|
| `Liquid_Level_mm` | Radar liquid level (mm) — target |
| `Cabin_Pressure_MPa` | Cabin pressure (MPa) |
| `Temp_Top_C` | Top temperature (°C) |
| `Temp_Mid_C` | Middle temperature (°C) |
| `Temp_Bot_C` | Bottom temperature (°C) |

If you use this code, please cite the paper.


In [ ]:
import copy
import time
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
DATA_PATH = "data/LPG_dataset.xlsx"   # replace with your own path
SAVE_DIR  = Path("results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "data_path": DATA_PATH,
    "seq_len": 30,
    "batch_size": 64,
    "epochs": 300,
    "lr": 1e-4,
    "d_model": 128,
    "nhead": 8,
    "num_layers": 4,
    "dropout": 0.05,
    "sensor_dropout_p": 0.0,
    "patience": 50,
    "alpha": 0.05,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "cti_w1": 0.4,
    "cti_w2": 0.3,
    "cti_w3": 0.3,
}

print(f"Device: {CONFIG['device']} | CP level: {(1-CONFIG['alpha'])*100:.0f}%")

## 1. Data loading and physics features

In [ ]:
def load_and_process_data(file_path):
    df = pd.read_excel(file_path)
    df.columns = [c.strip() for c in df.columns]

    df["P_abs"] = df["Cabin_Pressure_MPa"] + 0.1013
    df["T_gas_K"] = df["Temp_Top_C"] + 273.15
    df["Gas_Factor"] = df["T_gas_K"] / (df["P_abs"] + 1e-6)
    df["Delta_T_Vertical"] = df["Temp_Top_C"] - df["Temp_Bot_C"]
    df["Temp_Gradient"] = (
        (df["Temp_Top_C"] - df["Temp_Mid_C"])
        / (df["Temp_Mid_C"] - df["Temp_Bot_C"] + 1e-6)
    )

    feature_cols = [
        "Liquid_Level_mm", "Cabin_Pressure_MPa",
        "Temp_Top_C", "Temp_Mid_C", "Temp_Bot_C",
        "Delta_T_Vertical", "Temp_Gradient", "Gas_Factor",
    ]
    return df[feature_cols].dropna().values, feature_cols

In [ ]:
def create_dataset(data, seq_len):
    X, X_phys, Y = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        X_phys.append(data[i + seq_len - 1, -1])
        Y.append(data[i + seq_len, 0])
    return np.array(X), np.array(X_phys).reshape(-1, 1), np.array(Y).reshape(-1, 1)


def prepare_data_split(train, val, test, seq_len, batch_size):
    Xtr, Ptr, Ytr = create_dataset(train, seq_len)
    Xva, Pva, Yva = create_dataset(val,   seq_len)
    Xte, Pte, Yte = create_dataset(test,  seq_len)

    def make(loader_data, shuffle):
        X, Y, P = loader_data
        ds = TensorDataset(torch.Tensor(X), torch.Tensor(Y), torch.Tensor(P))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    return (
        make((Xtr, Ytr, Ptr), True),
        make((Xva, Yva, Pva), False),
        make((Xte, Yte, Pte), False),
        Pte,
    )

## 2. Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


class SensorDropout(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.training and self.p > 0:
            mask = torch.rand(x.size(0), x.size(1), 1, device=x.device) > self.p
            fmask = torch.ones_like(x)
            fmask[:, :, 0] = mask.squeeze(-1)
            return x * fmask
        return x


class PhysicsConstraintLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, gas_factor):
        return self.mlp(gas_factor)


class PhysicsResidualTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout, sensor_dropout_p):
        super().__init__()
        self.phys_adapter = PhysicsConstraintLayer()
        self.sensor_dropout = SensorDropout(p=sensor_dropout_p)
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.dropout_layer = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, x_phys):
        base = self.phys_adapter(x_phys)
        x = self.sensor_dropout(x)
        feat = self.transformer(self.pos_encoder(self.input_proj(x)))
        residual = self.head(self.dropout_layer(feat[:, -1, :]))
        return base + residual

## 3. Conformal prediction

In [ ]:
class ConformalPredictor:
    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.q_hat = None

    def calibrate(self, y_true, y_pred):
        scores = np.abs(y_true - y_pred).flatten()
        n = len(scores)
        q_level = np.ceil((n + 1) * (1 - self.alpha)) / n
        self.q_hat = np.quantile(scores, q_level)
        print(f"Calibrated: n={n}, q_hat={self.q_hat:.2f}")

    def predict(self, y_pred):
        if self.q_hat is None:
            raise ValueError("Call calibrate() first.")
        return y_pred - self.q_hat, y_pred + self.q_hat

## 4. Training, evaluation, and CTI

In [ ]:
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)


def get_predictions(model, loader, scaler_y, device):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for bx, by, bp in loader:
            bx, bp = bx.to(device), bp.to(device)
            preds.append(model(bx, bp).cpu().numpy())
            truths.append(by.numpy())
    preds = np.concatenate(preds, 0)
    truths = np.concatenate(truths, 0)
    return scaler_y.inverse_transform(preds), scaler_y.inverse_transform(truths)


def calculate_metrics(y_true, y_pred, lower, upper, confidence=0.95):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    wmape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100
    picp = np.mean((y_true >= lower) & (y_true <= upper)) * 100
    mpiw = np.mean(upper - lower)
    eta = 50
    cwc = mpiw * (1 + np.exp(eta * (confidence - picp / 100))) if picp < confidence * 100 else mpiw
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "WMAPE": wmape,
            "PICP": picp, "MPIW": mpiw, "CWC": cwc}

In [ ]:
def calculate_cti(y_true, y_pred, lower, upper, phys_true, phys_pred,
                  alpha=0.05, w1=0.4, w2=0.3, w3=0.3):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    y_range = np.max(y_true) - np.min(y_true)
    nrmse = rmse / (y_range + 1e-8)
    s_acc = 1 / (1 + nrmse)

    rho_true, _ = spearmanr(y_true.flatten(), phys_true.flatten())
    rho_pred, _ = spearmanr(y_pred.flatten(), phys_pred.flatten())
    s_phys = 1 - np.abs(rho_true - rho_pred)

    gamma = np.mean((y_true >= lower) & (y_true <= upper))
    w_norm = np.mean(upper - lower) / (y_range + 1e-8)
    s_uq = gamma * (1 - w_norm)

    cti = w1 * s_acc + w2 * s_phys + w3 * s_uq
    return {"CTI": cti, "S_acc": s_acc, "S_phys": s_phys, "S_uq": s_uq,
            "NRMSE": nrmse, "rho_true": rho_true, "rho_pred": rho_pred,
            "gamma": gamma, "gamma_target": 1 - alpha, "W_norm": w_norm}

In [ ]:
def plot_results(history, pred, truth, lower, upper):
    fig = plt.figure(figsize=(15, 10))

    ax = plt.subplot(3, 2, 1)
    ax.plot(history["train_loss"], label="Train")
    ax.plot(history["val_loss"], label="Val")
    ax.set(title="Training Loss", xlabel="Epoch", ylabel="MSE")
    ax.legend(); ax.grid(alpha=0.3)

    ax = plt.subplot(3, 2, 2)
    widths = (upper - lower).flatten()
    ax.hist(widths, bins=50, alpha=0.7, edgecolor="black")
    ax.axvline(np.mean(widths), color="red", ls="--", label=f"Mean: {np.mean(widths):.1f} mm")
    ax.set(title="CP Interval Width", xlabel="Width (mm)", ylabel="Frequency")
    ax.legend(); ax.grid(alpha=0.3)

    ax = plt.subplot(3, 2, 3)
    n = min(500, len(pred))
    ax.plot(truth[:n], label="True", alpha=0.7)
    ax.plot(pred[:n], label="Predicted", alpha=0.7)
    ax.fill_between(range(n), lower[:n].flatten(), upper[:n].flatten(),
                    alpha=0.2, label="95% CP")
    ax.set(title="Prediction with CP Interval", xlabel="Time", ylabel="Liquid Level (mm)")
    ax.legend(); ax.grid(alpha=0.3)

    ax = plt.subplot(3, 2, 4)
    ax.scatter(truth, pred, alpha=0.3, s=5)
    ax.plot([truth.min(), truth.max()], [truth.min(), truth.max()], "r--", label="Perfect")
    ax.set(title="Predicted vs True", xlabel="True (mm)", ylabel="Predicted (mm)")
    ax.legend(); ax.grid(alpha=0.3)

    ax = plt.subplot(3, 2, 5)
    window = 100
    cov = []
    for i in range(0, len(pred) - window, 10):
        hits = (truth[i:i+window] >= lower[i:i+window]) & (truth[i:i+window] <= upper[i:i+window])
        cov.append(np.mean(hits) * 100)
    ax.plot(cov)
    ax.axhline(95, color="red", ls="--", label="Target 95%")
    ax.set(title="Rolling Coverage", xlabel="Window", ylabel="Coverage (%)")
    ax.legend(); ax.grid(alpha=0.3)

    ax = plt.subplot(3, 2, 6)
    ax.hist((pred - truth).flatten(), bins=50, alpha=0.7, edgecolor="black")
    ax.axvline(0, color="red", ls="--")
    ax.set(title="Error Distribution", xlabel="Error (mm)", ylabel="Frequency")
    ax.grid(alpha=0.3)

    plt.tight_layout(); plt.show()


def plot_cti(cti):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cats = ["Accuracy", "Physics", "UQ"]
    vals = [cti["S_acc"], cti["S_phys"], cti["S_uq"]]

    angles = np.linspace(0, 2 * np.pi, len(cats), endpoint=False).tolist()
    vals_loop = vals + vals[:1]
    angles_loop = angles + angles[:1]
    ax1 = plt.subplot(121, projection="polar")
    ax1.plot(angles_loop, vals_loop, "o-", linewidth=2, label=f"CTI = {cti['CTI']:.3f}")
    ax1.fill(angles_loop, vals_loop, alpha=0.25)
    ax1.set_xticks(angles); ax1.set_xticklabels(cats)
    ax1.set_ylim(0, 1); ax1.set_title("CTI Radar")
    ax1.legend(loc="upper right"); ax1.grid(True)

    ax2 = axes[1]
    bars = ax2.bar(cats, vals, color=["#2E86AB", "#A23B72", "#F18F01"],
                   alpha=0.7, edgecolor="black")
    ax2.axhline(0.85, color="green", ls="--", label="Excellence (0.85)")
    ax2.set_ylim(0, 1); ax2.set_ylabel("Score")
    ax2.set_title(f"CTI Components (Total: {cti['CTI']:.3f})")
    ax2.legend(); ax2.grid(axis="y", alpha=0.3)
    for b in bars:
        ax2.text(b.get_x() + b.get_width() / 2, b.get_height(),
                 f"{b.get_height():.3f}", ha="center", va="bottom", fontweight="bold")

    plt.tight_layout(); plt.show()

## 5. Run experiment

In [ ]:
def main():
    raw, feature_cols = load_and_process_data(CONFIG["data_path"])
    n = len(raw)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    train_raw, val_raw, test_raw = raw[:train_end], raw[train_end:val_end], raw[val_end:]
    print(f"Split -> Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")

    scaler = MinMaxScaler().fit(train_raw)
    scaler_y = MinMaxScaler().fit(train_raw[:, 0:1])
    train_n, val_n, test_n = scaler.transform(train_raw), scaler.transform(val_raw), scaler.transform(test_raw)

    train_loader, val_loader, test_loader, test_phys = prepare_data_split(
        train_n, val_n, test_n, CONFIG["seq_len"], CONFIG["batch_size"]
    )

    device = CONFIG["device"]
    model = PhysicsResidualTransformer(
        input_dim=len(feature_cols),
        d_model=CONFIG["d_model"], nhead=CONFIG["nhead"],
        num_layers=CONFIG["num_layers"], dropout=CONFIG["dropout"],
        sensor_dropout_p=CONFIG["sensor_dropout_p"],
    ).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=10, factor=0.5)
    stopper = EarlyStopping(patience=CONFIG["patience"])
    history = {"train_loss": [], "val_loss": []}
    loss_fn = nn.MSELoss()
    t0 = time.time()

    for epoch in range(CONFIG["epochs"]):
        model.train()
        tr = 0
        for bx, by, bp in train_loader:
            bx, by, bp = bx.to(device), by.to(device), bp.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(bx, bp), by)
            loss.backward(); optimizer.step()
            tr += loss.item()
        tr /= len(train_loader)

        model.eval()
        va = 0
        with torch.no_grad():
            for bx, by, bp in val_loader:
                bx, by, bp = bx.to(device), by.to(device), bp.to(device)
                va += loss_fn(model(bx, bp), by).item()
        va /= len(val_loader)

        history["train_loss"].append(tr); history["val_loss"].append(va)
        if (epoch + 1) % 5 == 0 or epoch < 5:
            print(f"Epoch {epoch+1:03d} | Train: {tr:.5f} | Val: {va:.5f}")
        scheduler.step(va)
        stopper(va, model)
        if stopper.early_stop:
            print(f"Early stop at epoch {epoch+1}")
            break

    stopper.restore(model)
    train_time = time.time() - t0

    # CP calibration on validation set
    val_pred, val_truth = get_predictions(model, val_loader, scaler_y, device)
    cp = ConformalPredictor(alpha=CONFIG["alpha"])
    cp.calibrate(val_truth, val_pred)

    # Test
    test_pred, test_truth = get_predictions(model, test_loader, scaler_y, device)
    lower, upper = cp.predict(test_pred)

    metrics = calculate_metrics(test_truth.flatten(), test_pred.flatten(),
                                lower.flatten(), upper.flatten())

    # Denormalize physics proxy for CTI
    phys_scaler = MinMaxScaler().fit(test_raw[:, -1].reshape(-1, 1))
    phys_denorm = phys_scaler.inverse_transform(test_phys)

    cti = calculate_cti(test_truth, test_pred, lower, upper,
                        phys_denorm, phys_denorm.copy(),
                        alpha=CONFIG["alpha"],
                        w1=CONFIG["cti_w1"], w2=CONFIG["cti_w2"], w3=CONFIG["cti_w3"])

    print("\n=== Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("\n=== CTI ===")
    for k, v in cti.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    plot_results(history, test_pred, test_truth, lower, upper)
    plot_cti(cti)

    return {
        **metrics,
        "history": history, "train_time": train_time,
        "y_true": test_truth, "y_pred": test_pred,
        "lower": lower, "upper": upper,
        "phys_param": phys_denorm, "cti": cti,
    }


results = main()

## 6. Save results

In [ ]:
run_name = "PRT-CP_LPG"

save = {
    "name": run_name,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": {k: float(results[k]) for k in ["R2", "RMSE", "MAE", "PICP", "MPIW", "CWC", "WMAPE"]},
    "cti": results["cti"],
    "history": results["history"],
    "train_time": float(results["train_time"]),
    "config": CONFIG,
}

with open(SAVE_DIR / f"{run_name}_results.pkl", "wb") as f:
    pickle.dump(save, f)

pd.DataFrame({
    "y_true": results["y_true"].flatten(),
    "y_pred": results["y_pred"].flatten(),
    "lower":  results["lower"].flatten(),
    "upper":  results["upper"].flatten(),
    "phys":   results["phys_param"].flatten(),
}).to_csv(SAVE_DIR / f"{run_name}_predictions.csv", index=False)

print(f"Saved to {SAVE_DIR}/")

## CTI definition

The Composite Trustworthiness Index combines three dimensions:

- **Accuracy**: $S_{acc} = 1 / (1 + \text{NRMSE})$
- **Physics consistency**: $S_{phys} = 1 - |\rho_{true} - \rho_{pred}|$ (Spearman rank correlation against the physical proxy variable)
- **UQ reliability**: $S_{uq} = \gamma \cdot (1 - W_{norm})$, where $\gamma$ is empirical coverage and $W_{norm}$ is the mean interval width normalized by the data range

Final score: $\text{CTI} = w_1 S_{acc} + w_2 S_{phys} + w_3 S_{uq}$ with default weights $(0.4, 0.3, 0.3)$.